In [1]:
from glob import glob
import pandas as pd

In [2]:
prs_files = glob('../PolyGenie/profiles_data/GCATcore-*-EUR.profiles')

In [3]:
prs_files[0]

'../PolyGenie/profiles_data/GCATcore-mono-EUR.profiles'

In [14]:
for prs in prs_files:
    df = pd.read_csv(prs, sep="\s+")
    tokeep = [x for x in df.columns if '*' in x]
    print(tokeep)
    for col in tokeep:
        trait = col.split('_ldak')[0]
        tmp = df[['IID', col]]
        tmp.columns = ['ID', 'PRS']
        tmp.to_csv(f'data/prs/{trait}.profiles', sep='\t', index=None)

['mono_ldak_Model198*', 'mono_perc_ldak_Model154*']
['covid_r7_c2_ldak_Model153*']
['loss_x_binary_ldak_Model147*']
['atrialfib_ldak_Model144*']
['lc_n2_ldak_Model10*']
['anterior_uveitis_ldak_Model122*']
['height_ldak_Model214*']
['mc_hemo_ldak_Model126*']
['psoriasis_ldak_Model109*']
['restless_leg_ldak_Model167*']
['covid_r7_b2_ldak_Model185*']
['reti_count_ldak_Model154*']
['diet_carb_ldak_Model198*']
['mono_perc_ldak_Model154*']
['subcortical_ldak_Model197*']
['ltl_ldak_Model166*']
['hypothyroidism_ldak_Model188*']
['caudalmiddlefrontal_cortical_ldak_Model191*']
['amyloid_p_ldak_Model154*']
['eos_perc_ldak_Model161*', 'eos_ldak_Model154*']
['diet_fat_ldak_Model9*']
['vegetarianism_ldak_Model12*']
['gestational_diabetes_ldak_Model193*']
['fvc_ldak_Model194*']
['whr_ldak_Model146*']
['an_ldak_Model12*']
['global_cortical_ldak_Model12*']
['pef_ldak_Model186*']
['diver_ldak_Model195*']
['broad_depression_ldak_Model173*']
['raynaud_ldak_Model218*']
['covid_r7_a2_ldak_Model154*']
['hand

In [10]:
## Prepare ICD codes and phecodes

phecodes = pd.read_csv('../PolyGENIE_data/3-phenotypes/1-phecodes/EHR_phecodes.csv', sep=';')
phecodes = phecodes[~phecodes['PHECODE'].isna()]

In [12]:
consulta = pd.read_csv('/home/labs/dnalab/xfarrer/lims/mysql/output/consulta_nivell2.csv', sep=';')
consulta = consulta[consulta['SPAIN_CORE'] == True]

/tmp/ipykernel_3942749/752963782.py:1: DtypeWarning: Columns (31,48,50,52,54,59,61,63,76,77,79,82,83,89,90,91,94,96,102,107,108) have mixed types. Specify dtype option on import or set low_memory=False.
  consulta = pd.read_csv('/home/labs/dnalab/xfarrer/lims/mysql/output/consulta_nivell2.csv', sep=';')


In [13]:
ent2id = dict(zip(consulta['entity_id'], consulta['GENOTYPED_SAMPLE']))

In [14]:
phecodes['ID'] = phecodes['entity_id'].map(ent2id)
phecodes = phecodes[~phecodes['ID'].isna()]
phecodes = phecodes[~phecodes['PHECODE'].isna()]

In [15]:
phecodes['PHECODE_SIMPLE'] = phecodes['PHECODE'].map(lambda x: str(x).split('.')[0])

In [19]:
phecodes.sort_values('EDAT')

,entity_id,DATA,EDAT,ICD10,PHECODE,ID,PHECODE_SIMPLE
218412,=E00251514579321,1965-02-28,1.0,T300,1000.0,AAA4272085,1000
264299,=E00251524999221,1963-07-08,1.0,Q667,755.1,AAA4216225,755
198644,=E00251513270121,1971-01-01,1.0,Q249,747.1,AAA3258905,747
294634,=E00251529098921,1976-09-29,1.0,G809,343.0,AAA4625922,343
169908,=E00251510692521,1960-06-07,1.0,G40909,345.0,AAA3884112,345
...,...,...,...,...,...,...,...
90019,=E00251427202821,2023-04-11,74.0,R05,512.8,AAA1604036,512
122945,=E00251429906621,2023-10-20,75.0,H259,366.2,AAA3233160,366
684018,=E00251429906621,2023-10-27,75.0,T783XXA,946.0,AAA3233160,946
104978,=E00251429180321,2023-10-06,75.0,R54,290.0,AAA1862548,290


In [36]:
phecodes[['PHECODE_SIMPLE', 'ID']]

,PHECODE_SIMPLE,ID
1,41,AAA4031859
3,327,AAA4629679
11,318,AAA4018234
14,727,AAA3257172
21,300,AAA1862881
...,...,...
1572411,474,AAA4624336
1572412,939,AAA1604007
1572420,745,AAA4833810
1572421,300,AAA4833810


In [48]:
# Create the binary matrix
phecode_matrix = pd.crosstab(phecodes['ID'], phecodes['PHECODE_SIMPLE'])

# Ensure values are 1/0 instead of counts
phecode_matrix[phecode_matrix > 0] = 1

# Reset index if you want ID as a column
phecode_matrix.reset_index(inplace=True)

In [51]:
phecode_matrix.to_csv('../polygenie-pipeline/data/phenotypes/phecodes.csv', sep=';', index=None)

In [59]:
metabolites = pd.read_csv('../PolyGENIE_data/0-data/metabolites.tsv', sep='\t')
metabolites.columns = ['ID'] + metabolites.columns.tolist()[1:]
metabolites.to_csv('../polygenie-pipeline/data/phenotypes/metabolites.csv', sep=';', index=None)

In [68]:
icd = pd.read_csv('../PolyGenie/input_data/phenotypes.tsv', sep='\t')

/tmp/ipykernel_1221684/3458436621.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  icd = pd.read_csv('../PolyGenie/input_data/phenotypes.tsv', sep='\t')


In [71]:
icd = icd[icd['code'] == 'ICD10']

In [72]:
consulta['entity_id'] = consulta['entity_id'].str.lstrip('=')
ent2id = dict(zip(consulta['entity_id'], consulta['GENOTYPED_SAMPLE']))

In [75]:
icd['ID'] = icd['entity_id'].map(ent2id)
icd = icd[~icd['ID'].isna()]
icd = icd[~icd['code'].isna()]

/tmp/ipykernel_1221684/3849224439.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  icd['ID'] = icd['entity_id'].map(ent2id)


In [79]:
# Create the binary matrix
icd_matrix = pd.crosstab(icd['ID'], icd['disease'])

# Ensure values are 1/0 instead of counts
icd_matrix[icd_matrix > 0] = 1

# Reset index if you want ID as a column
icd_matrix.reset_index(inplace=True)

In [82]:
icd_matrix.to_csv('../polygenie-pipeline/data/phenotypes/icd_codes.csv', sep=';', index=None)

In [81]:
icd[icd['disease'] == '???']

,entity_id,disease,code,ID
162523,E00251514783921,???,ICD10,AAA4275945


In [88]:
quest = pd.read_csv('../PolyGENIE_data/3-phenotypes/2-questionnaire/phenotype.csv', sep='\t')
col2keep = quest.columns.tolist()[1:]
quest['ID'] = quest['entity_id'].map(ent2id)
quest = quest[~quest['ID'].isna()]
quest = quest[['ID'] + col2keep]

In [90]:
quest.to_csv('../polygenie-pipeline/data/phenotypes/questionnaire.csv', sep=';', index=None)

In [87]:
metabolites

,ID,Lactic_acid,Alanine,Glycine,2_Hydroxybutanoic_acid,3_Hydroxybutanoic_acid,Valine,Leucine,Glycerol,Isoleucine,...,Large_HDL_P,Medium_HDL_P,Small_HDL_P,VLDL_Z,LDL_Z,HDL_Z,Non_HDL_P,Total_P_HDL_P,LDL_P_HDL_P,Leucine_Isoleucine
0,AAA4226491,0.607000,-1.080112,1.081015,-0.850432,-0.297354,-0.969368,-0.795288,0.108302,-0.985619,...,0.140297,0.398577,0.266417,-0.958941,-0.362818,-0.012346,-0.778121,-0.793215,-0.790457,0.734519
1,AAA4227554,1.337121,0.029988,0.003779,0.040072,1.685338,-2.124562,-0.990545,0.809891,-0.377929,...,0.856234,0.234696,-0.354756,0.709018,-0.091083,1.378935,0.312664,0.297881,0.400215,-2.036855
2,AAA4272060,0.188314,-0.198583,1.157366,0.312664,-0.158642,0.620995,-0.019402,-0.464404,0.237805,...,-0.180113,-0.076418,-1.089186,-1.025726,0.019906,1.329766,-0.625891,-0.081473,0.008819,-0.886435
3,AAA3233789,0.345648,-0.590722,-1.290538,-0.826795,-0.563342,-0.135210,0.202181,-0.114894,0.087542,...,1.867957,-0.306849,0.777439,-0.421599,-0.112358,-0.847542,0.233661,-0.064294,-0.092096,0.454324
4,AAA4227567,1.438742,1.681182,-0.734519,1.730842,1.302214,0.247666,0.471714,0.771318,0.583537,...,-2.350533,-0.486976,1.067555,0.326452,-2.429302,-2.170031,0.178577,-0.144369,-0.386603,-0.535796
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4983,AAA4833043,-0.835337,-1.431685,-0.745785,-1.524813,1.807954,-0.048648,0.413901,0.538125,0.871604,...,-1.448744,1.448744,-0.597335,0.976649,0.194473,2.429302,0.607606,0.387689,0.461600,-1.430282
4984,AAA4833009,-0.113373,1.266575,2.857346,-0.653094,-1.453075,0.389320,0.440956,1.829017,0.364433,...,-0.231590,0.331240,0.281590,2.124562,-0.054198,0.313193,1.552802,1.138859,1.120733,0.417197
4985,AAA4833071,-1.296354,0.876773,0.854055,-0.608212,-0.951783,0.184724,0.030492,-0.687795,0.110837,...,-0.539873,0.549810,-0.543958,-0.251825,0.624665,1.037738,-1.362178,-1.002134,-0.992192,-0.075408
4986,AAA4833467,-1.845364,-0.475660,-0.919772,-1.592721,-1.054285,-0.045621,-0.291566,0.624665,-0.142333,...,0.522465,0.638815,0.706429,1.728593,-0.202696,0.201667,1.672957,1.007142,0.925943,-0.260678


In [1]:
### Prepare variables to upload to ddbb
import pandas as pd
import numpy as np


In [2]:
cons = pd.read_csv('/home/labs/dnalab/xfarrer/lims/mysql/output/consulta_nivell2.csv', sep=';')
phecode = pd.read_csv('data/phenotypes/phecodes.csv', sep=';')
icd = pd.read_csv('data/phenotypes/icd_codes.csv', sep=';')
df = pd.read_csv('../PolyGenie/input_data/individuals.tsv', sep='\t')

/tmp/ipykernel_3942749/596823946.py:1: DtypeWarning: Columns (31,48,50,52,54,59,61,63,76,77,79,82,83,89,90,91,94,96,102,107,108) have mixed types. Specify dtype option on import or set low_memory=False.
  cons = pd.read_csv('/home/labs/dnalab/xfarrer/lims/mysql/output/consulta_nivell2.csv', sep=';')


In [6]:
df

,entity_id,iid,gender,age,bmi,self_perceived_hs,cohort
0,E00251413396721,AAA1604436,FEMALE,55.0,23.27,Very good,gcat
1,E00251413396921,AAA3231295,MALE,61.0,32.59,Good,gcat
2,E00251413398721,AAA3253227,FEMALE,52.0,18.06,Very good,gcat
3,E00251413469721,AAA3233779,FEMALE,75.0,41.53,Fair,gcat
4,E00251413472721,AAA4018038,FEMALE,52.0,23.99,Good,gcat
...,...,...,...,...,...,...,...
4983,E00251628109621,AAA4833064,FEMALE,63.0,20.57,Good,gcat
4984,E00251628110521,AAA4833088,FEMALE,69.0,38.61,Fair,gcat
4985,E00251629151321,AAA4632450,MALE,64.0,27.03,Good,gcat
4986,E00251629152721,AAA4632462,MALE,56.0,25.58,Good,gcat


In [9]:
df['age'].min()

np.float64(48.0)

In [7]:
def compute_age_distribution(df, target_id, df_pheno):
    ind = df_pheno[df_pheno[target_id] == 1]['ID'].tolist()
    df = df[df['iid'].isin(ind)].copy()

    df["age_group"] = pd.cut(
        df["age"],
        bins=range(0, 101, 5),
        right=False
    )

    # Male / Female
    base = (
        df.groupby(["age_group", "gender"])
          .size()
          .reset_index(name="count")
    )

    # Combined
    combined = (
        base.groupby("age_group", observed=True)["count"]
            .sum()
            .reset_index()
    )
    combined["gender"] = "Combined"

    out = pd.concat([base, combined])
    out["variable"] = "age"
    out["category"] = out["age_group"].astype(str)
    out["target_id"] = target_id

    return out[["target_id", "variable", "category", "gender", "count"]]

In [8]:
compute_age_distribution(df, 'I10', icd)

/tmp/ipykernel_3942749/2371079061.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(["age_group", "gender"])


,target_id,variable,category,gender,count
0,I10,age,"[0, 5)",FEMALE,0
1,I10,age,"[0, 5)",MALE,0
2,I10,age,"[5, 10)",FEMALE,0
3,I10,age,"[5, 10)",MALE,0
4,I10,age,"[10, 15)",FEMALE,0
5,I10,age,"[10, 15)",MALE,0
6,I10,age,"[15, 20)",FEMALE,0
7,I10,age,"[15, 20)",MALE,0
8,I10,age,"[20, 25)",FEMALE,0
9,I10,age,"[20, 25)",MALE,0
